In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import*
from pyspark.sql.functions import *


from delta import *

In [ ]:
spark = ( 
    SparkSession
    .builder
    .master("local[*]")
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate() 
)

In [ ]:
studentsSchema = (
    StructType([
        StructField("ID_ALUNO",     IntegerType(),True),
        StructField("NOME_ALUNO",   StringType(),True),
        StructField("GRADE",             StringType(),True),
        StructField("CPF",         StringType(),True)
    ])
)

In [ ]:
studentsData = [
    (1, "Luis", "FUNDAMENTAL", "564.654.654-21"),
    (2, "Jonh", "POS-GRADUACAO", "321.497.312-21"),
    (3, "Hanz", "MEDIO", "978.348.914-45")
]

# Criando e Salvando a delta table

In [ ]:
df = spark.createDataFrame(data=studentsData,schema=studentsSchema)

df.show(truncate=False)

In [ ]:
( 
    df
    .write
    .format("delta")
    .mode('overwrite')
    .save("./RAW/ESTUDANTES")
)

# Novos Dados

In [ ]:
new_students = [
    (1,"Luis","MEDIO", "564.654.654-21"),
    (2, "Ana", "POS-GRADUACAO", "321.497.312-21"),
    (4, "Jonas", "CRECHE", "621.645.312.87")
]

df_new = spark.createDataFrame(data=new_students, schema=studentsSchema)

df_new.show()

# Merge

In [ ]:
deltaTable = DeltaTable.forPath(spark, "./RAW/ESTUDANTES")

(
    deltaTable.alias("dados_atuais")
    .merge(
        df_new.alias("novos_dados"),
        "dados_atuais.ID_ALUNO = novos_dados.ID_ALUNO"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

deltaTable.toDF().show()

# Excluir

In [ ]:
deltaTable.delete(col("NOME_ALUNO") == "Luis")

deltaTable.toDF().show()

# Update

In [ ]:
deltaTable.update(
    condition = col("NOME_ALUNO") == "Ana",
    set = { "GRADE": lit("FORMADA") } )

deltaTable.toDF().show()